# **Tutorial 4** $\cdot$ Uncertainty Quantification

> **Abstract.** This tutorial introduces *uncertainty quantification (UQ)* for molecular property prediction. Unlike attribution or counterfactual methods, UQ methods do not explain *why* a model made a given prediction &mdash; instead, they quantify *how confident* the model is in that prediction. We demonstrate **deep ensembles with mean-variance estimation (MVE)**: training several independent neural networks &mdash; each of which predicts not only the target value but also its own per-input variance &mdash; on slightly different subsets of the data, and then combining their outputs through a moment-matched mixture-of-Gaussians aggregation that cleanly separates *aleatoric* and *epistemic* uncertainty. The MVE-specific training tricks (β-NLL loss, gradient clipping, exponential variance activation) follow the [`truthful_counterfactuals`](https://github.com/the16thpythonist/truthful_counterfactuals) reference implementation. Using the AqSolDB water solubility dataset as a case study, we train an ensemble of graph neural networks, then evaluate the quality of the resulting uncertainty estimates in two complementary ways: (1) calibration on in-distribution data, where uncertainty should correlate with prediction error; and (2) detection of distribution shift, where uncertainty should rise as the test molecules become progressively more dissimilar from the training distribution. To this end we construct three out-of-distribution (OOD) tiers of increasing severity &mdash; from larger drug-like molecules through unseen chemical elements all the way to large polycyclic aromatic hydrocarbons (PAHs) far outside the chemistry the model has ever seen.


**💾 Datasets.** This notebook uses two datasets. The primary dataset is the [AqSolDB](https://www.kaggle.com/datasets/sorkun/aqsoldb-a-curated-aqueous-solubility-dataset) dataset introduced by [Sorkun et al.](https://www.nature.com/articles/s41597-019-0151-1) &mdash; the same regression benchmark used in Tutorials 1 and 3 &mdash; on which the ensemble is trained. To probe out-of-distribution behaviour we additionally use a small auxiliary set of ~50 polycyclic aromatic hydrocarbons (PAHs) (`load_dataset_pahs`) shipped with this package, together with a hand-curated SMILES list defined inline in the notebook covering unseen chemical elements.

**📦 Packages.** Packages used in this notebook include:
- [RDKit](https://www.rdkit.org/docs/index.html) Cheminformatics framework for processing of SMILES strings.
- [Pandas](https://pandas.pydata.org/docs/index.html) for data representation and processing.
- [Pytorch](https://docs.pytorch.org/docs/stable/index.html) and [Pytorch Lightning](https://lightning.ai/docs/pytorch/stable/) for the training of the individual ensemble members.
- [Pytorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/) framework for the construction of the graph neural network ensemble.
- [NetworkX](https://networkx.org/) for the intermediate molecular graph representation.

In [39]:
import os
import copy
import random
import warnings

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import rdkit.Chem as Chem
import lightning.pytorch as pl
import torch.nn as nn
import torch.nn.functional as F
from rich.pretty import pprint
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from torch.nn.utils.parametrizations import spectral_norm
from torchmetrics import MeanSquaredError, R2Score

warnings.filterwarnings('ignore')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

from xai_chem_review import load_dataset_aqsoldb
from xai_chem_review import load_dataset_pahs

# Fix the random seeds for reproducibility of the train-test split and the
# ensemble subsampling. The remaining randomness in ensemble member training
# (weight initialisation, batch shuffling) is what gives the ensemble its diversity.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

The `load_dataset_aqsoldb` function loads and returns the AqSolDB dataset into a pandas `DataFrame` object. Each molecule is represented as a [SMILES](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) string with a continuous water solubility value (logS).

In [40]:
data_frame: pd.DataFrame = load_dataset_aqsoldb()
print(f'Loaded dataset with {data_frame.shape[0]} rows')
data_frame.head()

Loaded dataset with 11024 rows


,index,ID,smiles,InChIKey,solubility,split
0,0,A-10,Cc1cccc(C=C)c1,JZHGRUMIRATHIU-UHFFFAOYSA-N,-3.123150,1
1,1,A-100,Cc1cc(cc(C)c1O)C(C)(C)c2cc(C)c(O)c(C)c2,ODJUOZPKKHIEOZ-UHFFFAOYSA-N,-4.952869,1
2,2,A-1000,O=C1CCCCCCCCCOCCCCCO1,MKEIDVFLAWJKMY-UHFFFAOYSA-N,-3.883849,1
3,3,A-1002,CCCCCCCCCCC(C)CCCCCCCC,FFVPRSKCTDQLBP-UHFFFAOYSA-N,-6.451105,1
4,4,A-1003,NC(=O)N=NC(N)=O,XOZUGNYVDXMRKW-UHFFFAOYSA-N,-3.546243,1


In [ ]:
data_frame['mol'] = data_frame['smiles'].apply(Chem.MolFromSmiles)
data_frame = data_frame.dropna(subset=['mol']).reset_index(drop=True)
print(f'After mol conversion {data_frame.shape[0]} rows remaining')

After mol conversion 11022 rows remaining


KeyboardInterrupt: 

## **4.1** $\cdot$ 💡 Uncertainty Quantification

The previous tutorials have introduced explanation methods that aim to make a model's predictions *interpretable* &mdash; either by attributing importance to features and substructures, or by suggesting structural modifications that would change the prediction. All of these methods share an implicit assumption: that the prediction itself is *trustworthy enough to be worth explaining*. In real-world deployment, however, machine-learning models frequently encounter inputs that look nothing like their training data &mdash; novel chemical scaffolds, or even entirely new classes of compounds. On such inputs models often produce confidently wrong predictions, and any explanation generated for such a prediction would explain a hallucination rather than a meaningful chemical insight.

*Uncertainty quantification (UQ)* methods address this gap. Rather than explaining a prediction, they attach a second number to it: an estimate of *how reliable* that prediction is likely to be. A useful uncertainty estimate should be *small* on inputs the model is well-equipped to handle &mdash; meaning predictions are close to the ground truth on average &mdash; and *large* on inputs where the model is operating outside its competence. In practice, this allows a downstream user to triage predictions: trust low-uncertainty estimates, double-check high-uncertainty ones, and reject those above some threshold entirely.

**Relevance to Molecular Property Prediction.** Chemical space is vast &mdash; far larger than any training set can hope to cover. A model trained on a curated dataset of small drug-like molecules has, in a meaningful sense, only seen a tiny corner of this space. When such a model is asked to predict properties of larger molecules, of compounds containing unusual atoms, or of polymers and biomolecules, it will still produce a number &mdash; but that number may be essentially arbitrary. Well-calibrated uncertainty estimates allow practitioners in drug discovery and materials science to recognise such situations *before* acting on the predictions.

**Aleatoric vs. Epistemic Uncertainty.** Conceptually, prediction uncertainty has two distinct sources. *Aleatoric* uncertainty arises from inherent noise in the data &mdash; e.g. experimental measurement error in the recorded solubility values. This source of uncertainty cannot be reduced by collecting more training data of the same kind. *Epistemic* uncertainty, in contrast, arises from limited knowledge of the model &mdash; regions of input space where the training data is sparse or absent. Epistemic uncertainty can in principle be reduced by training on more diverse data. The ensemble method introduced in this tutorial primarily captures *epistemic* uncertainty: it reveals where the model has not seen enough data to settle on a confident answer.

## **4.2** $\cdot$ 📚 *Deep Ensembles with Mean-Variance Estimation*

The conceptually simplest and empirically one of the most effective approaches to uncertainty quantification for neural networks was popularised by [Lakshminarayanan *et al.*](https://arxiv.org/abs/1612.01474) under the name *deep ensembles*. The original idea is straightforward: train an ensemble of several networks that solve the same task and use the *disagreement* between their predictions as an uncertainty estimate. We extend this approach in two important ways, both following the implementation of the [`truthful_counterfactuals`](https://github.com/the16thpythonist/truthful_counterfactuals) repository:

1. **Mean-Variance Estimation (MVE).** Each ensemble member predicts not just a scalar mean $\mu_{\theta_m}(\mathbf{x})$ but also a per-input variance $\sigma^2_{\theta_m}(\mathbf{x})$. The variance head is a small MLP on top of the same graph embedding the mean head uses, trained jointly with the mean head against a Gaussian negative log-likelihood (NLL) loss. This gives each individual member the ability to express its own *aleatoric* uncertainty &mdash; the irreducible noise it expects in the label &mdash; rather than relying solely on inter-member disagreement.

2. **Moment-matched aggregation.** When the $M$ ensemble members each output a Gaussian, the ensemble as a whole defines a *mixture of Gaussians* &mdash; which is not itself Gaussian. To recover scalar mean and variance for downstream use, we approximate the mixture by the single Gaussian with matching first and second moments (the law of total variance):

$$\mu_*(\mathbf{x}) \;=\; \frac{1}{M} \sum_m \mu_{\theta_m}(\mathbf{x}),
\qquad
\sigma^2_*(\mathbf{x}) \;=\; \frac{1}{M} \sum_m \!\bigl(\sigma^2_{\theta_m}(\mathbf{x}) + \mu^2_{\theta_m}(\mathbf{x})\bigr) \;-\; \mu^2_*(\mathbf{x}).$$

The total variance $\sigma^2_*$ decomposes cleanly into two interpretable components: the average within-network variance (the *aleatoric* part, from the individual regression heads) and the variance of the per-network means (the *epistemic* part, the spread of the ensemble). We will track both components separately throughout the analysis.

**Implementation.** Concretely, we train an ensemble of $N$ graph neural networks on AqSolDB. Each member is a *residual* GIN-based message-passing network (single input projection followed by residual GIN updates) with a separate variance head ending in an *exponential* activation (ensuring $\sigma^2 > 0$), every `nn.Linear` spectral-normalised so the whole forward map is 1-Lipschitz, and ReLU activations throughout. The spectral-normalisation + ReLU recipe makes the feature extractor *distance-aware* &mdash; the geometric motivation is explained in the info box below. To inject ensemble diversity along several independent axes, each member is trained on a different random 60% subsample of the training set, with independently sampled batch size, learning rate, hidden dimensionality and number of GNN layers. Each atom's feature vector additionally carries 32 random-walk return probabilities (the diagonals of $P^1, P^2, \ldots, P^{32}$ on the unweighted molecular graph), a graph-theoretic positional encoding that gives the GNN explicit information about each atom's local ring structure. We omit the optional MSE warm-up phase used by the reference implementation and train directly with the β-NLL loss from epoch 0 &mdash; relying on the β re-weighting (Seitzer et al., 2022), the exponential variance activation, the small numerical floor on $\sigma^2$, and persistent gradient clipping to keep the optimisation stable. These tricks are described in detail in the info box below.

<details style="border: 1.5px solid #536CCE; border-radius: 3px; padding: 10px; background-color:#EFF2FD; color: black; font-size: 0.9em;">
<summary style="cursor: pointer; font-weight: bold; color: #536CCE;">📔 Why Diversity Matters in Ensembles</summary>

The single most important property of a useful ensemble is that its members *make different mistakes*. If all $N$ networks in an ensemble converged to identical solutions, they would always agree, the ensemble standard deviation would always be zero, and there would be no uncertainty signal at all &mdash; the ensemble would behave as a single (slightly slower) model. The uncertainty signal we extract from a deep ensemble is precisely the *variance* of its members\' opinions, and this variance only exists if the members have been trained in ways that lead them to learn meaningfully different functions.

**Sources of Diversity.** A well-constructed ensemble injects randomness from several independent sources, each pulling members in different directions. This notebook exercises all five:

1. **Random weight initialisation.** Even when trained on identical data, neural networks initialised with different random seeds converge to different local minima in the highly non-convex loss landscape. This is the cheapest source of diversity and is what the original deep ensembles paper relied on. In this notebook each member is seeded independently via `pl.seed_everything`.

2. **Random data subsampling.** Each ensemble member is trained on a different random subset of the training data &mdash; here a 60%-subsample drawn without replacement. Lowering the subsample fraction (compared to the more usual 80%) emphasises this diversity axis at the cost of mildly reducing per-member accuracy.

3. **Optimisation-hyperparameter randomisation.** Each member receives a slightly different batch size and learning rate, drawn around a common mean. Different optimisation hyperparameters lead members to different points in the loss landscape even when the data and initialisation happen to be similar.

4. **Architectural randomisation.** Each member uses a slightly different GNN architecture, drawn from a small set of hidden-dimension and depth choices (`hidden_dim ∈ {48, 64, 80, 96, 112}`, `n_conv_layers ∈ {2, 3, 4}`). Architectural diversity is empirically one of the strongest individual axes for decorrelating ensemble members, because two members with different effective receptive fields will respond to OOD inputs in qualitatively different ways.

5. **Random batch ordering.** Even with the same data and initialisation, training with different mini-batch sequences produces different intermediate gradients and therefore different final parameters. This randomness comes "for free" from PyTorch\'s data loader.

**Why It Matters for OOD Detection.** On an in-distribution molecule that resembles many training examples, every reasonable model &mdash; regardless of which subsample, seed, learning rate or architecture it saw &mdash; converges on roughly the same prediction, because the data unambiguously supports one answer. On an out-of-distribution molecule the training data does *not* uniquely determine what the model should predict &mdash; and different members, biased differently by their training conditions, project their respective biases onto the unfamiliar input and produce different answers. The ensemble disagreement is therefore a direct read-out of how much the training data "underdetermines" the prediction at a given point &mdash; precisely the epistemic uncertainty we care about for distribution-shift detection.

**Practical Trade-off.** Increasing $N$ improves the statistical quality of the uncertainty estimate but linearly increases training and inference cost. Five to ten members is a common production sweet spot; this notebook defaults to $N = 5$.

</details>

<details style="border: 1.5px solid #536CCE; border-radius: 3px; padding: 10px; background-color:#EFF2FD; color: black; font-size: 0.9em;">
<summary style="cursor: pointer; font-weight: bold; color: #536CCE;">📔 Making MVE Training Actually Work: β-NLL Without Warm-up</summary>

Training a model to predict its own variance with a vanilla Gaussian NLL objective

$$\mathcal{L}_{\text{NLL}} \;=\; \frac{1}{N} \sum_{i} \frac{1}{2}\bigl( \log \sigma^2_i \;+\; (y_i - \mu_i)^2 / \sigma^2_i \bigr)$$

is notoriously fragile. The two failure modes most commonly observed are:

1. **Variance-collapse before the mean is accurate.** Early in training the predicted means are wildly wrong, so the only way the model can lower the NLL is to drive $\sigma^2$ to extremely large values to absorb the error term. Once the variance head has settled into "predict large $\sigma^2$ everywhere", the mean head receives almost no useful gradient signal and the model gets stuck.

2. **Gradient explosions in the NLL term.** The denominator $1 / \sigma^2$ in the loss makes the gradient with respect to small predicted variances arbitrarily large. A single optimisation step can then move the parameters into a regime where subsequent variance predictions are unstable.

One common safeguard against (1) is a **two-phase training schedule** &mdash; train the mean head with MSE first, then flip on the variance head later. The reference implementation in `truthful_counterfactuals` supports this via an `MveCallback`. We have **deliberately omitted that warm-up here** to keep the training loop simple: in our setting the remaining stabilisation tricks below are sufficient to converge cleanly from epoch 0. The warm-up remains a useful tool for harder problems or smaller datasets where the variance head genuinely cannot find traction without it.

The three tricks that *are* used in this notebook to make warmup-free MVE training behave well:

**Trick 1: β-NLL re-weighting (Seitzer et al., 2022).** Instead of the vanilla NLL above, we minimise

$$\mathcal{L}_{\beta\text{-NLL}} \;=\; \frac{1}{N} \sum_{i} \frac{1}{2}\,\bigl[\!\!\underbrace{\sigma_i^{2\beta}}_{\text{stop-grad}}\!\!\bigr] \cdot \bigl( \log \sigma^2_i \;+\; (y_i - \mu_i)^2 / \sigma^2_i \bigr).$$

The detached $\sigma^{2\beta}$ factor *re-weights* each sample's contribution by its own predicted variance, without itself flowing gradients. The original paper shows that this dramatically stabilises training compared to vanilla NLL while preserving the same optimal solution. We use $\beta = 0.5$, matching the reference implementation. The relevant detail is that the re-weighting is *detached*: it changes which samples dominate the loss but does not introduce additional gradient terms. Vanilla NLL implicitly down-weights samples by $1/\sigma^2$ so the optimiser focuses its capacity on samples it is already confident about &mdash; backwards. β-NLL with β=0.5 cancels that weighting almost completely.

**Trick 2: Exponential activation on the variance head (with the `+ 0` autograd workaround).** The variance head ends in `exp(·) + 0` rather than softplus. Exponential activations are smoother and produce stronger gradient signal for very small predicted variances, which empirically helps the variance head become more confident in regions where the mean head is accurate. The literal `+ 0` is an autograd workaround for a subtle PyTorch issue that occasionally zeroes out gradients through `torch.exp`; the reference implementation keeps it in place.

**Trick 3: Persistent gradient clipping by value.** Gradient clipping (by value, threshold 50) is on throughout training. Catches the rare-but-catastrophic exploding-gradient events that can otherwise undo an entire training run in a single step. By-value clipping (rather than by-norm) targets exactly the offending dimensions of the gradient vector, which under β-NLL tend to be one or two outlier samples with tiny $\sigma^2$.

Together with a small numerical floor (`eps = 1e-6`) on the predicted variance, these three tricks alone are enough to reliably train an MVE ensemble from epoch 0 to a model whose variance head produces a meaningful aleatoric signal &mdash; which we then combine with the epistemic disagreement across members.

</details>

<details style="border: 1.5px solid #536CCE; border-radius: 3px; padding: 10px; background-color:#EFF2FD; color: black; font-size: 0.9em;">
<summary style="cursor: pointer; font-weight: bold; color: #536CCE;">📔 Why Spectral Normalisation + ReLU? Distance-Aware Feature Extractors</summary>

The GNN in this notebook is wrapped with two architectural choices that look like minor implementation details but in fact directly shape the *uncertainty signal*: every `nn.Linear` is spectral-normalised, and every activation is a ReLU.

**The Lipschitz argument.** A function $f$ has Lipschitz constant $L$ if for every pair of inputs $\mathbf{x}, \mathbf{x}'$ we have $\|f(\mathbf{x}) - f(\mathbf{x}')\| \le L \cdot \|\mathbf{x} - \mathbf{x}'\|$. Lipschitz-bounded models cannot blow up small input differences into arbitrarily large output differences &mdash; or equivalently, two inputs that are far apart in input space are *guaranteed* to be at least proportionally far apart in feature space. This *distance-awareness* is exactly what an OOD detector wants: it makes it impossible for the encoder to accidentally map an OOD input to the same embedding as a familiar one.

**Where the bound comes from.** A composition of $L_i$-Lipschitz functions is $\prod_i L_i$-Lipschitz. For neural networks:

- A linear layer $\mathbf{W}\mathbf{x}$ has Lipschitz constant equal to the *largest singular value* of $\mathbf{W}$ &mdash; its *spectral norm*. Spectral normalisation rescales $\mathbf{W} \leftarrow \mathbf{W} / \sigma_{\max}(\mathbf{W})$ on every forward pass (using one or two power-iteration steps), forcing this norm to be 1.
- ReLU and other 1-Lipschitz pointwise activations (Identity, LeakyReLU with slope ≤ 1, tanh) have Lipschitz constant 1. GELU and Swish/SiLU are *not* 1-Lipschitz, which is why we replaced GELU/LeakyReLU with ReLU here.
- Message-passing aggregations like GIN's sum-then-MLP rule preserve Lipschitz boundedness as long as the MLP that follows the aggregation is itself Lipschitz-bounded &mdash; hence wrapping the GIN MLP's two `nn.Linear` layers with spectral norm. We use `GINConv` rather than `GCNConv` precisely because GINConv lets *us* supply the MLP from outside, so we can spectral-normalise its linears without monkey-patching internal PyG state.

Put together, *most* components of the network are 1-Lipschitz (spectral-normalised linears + ReLU). One subtlety: the encoder uses *residual updates* of the form $x \leftarrow x + \mathrm{ReLU}(\text{GIN}(x))$, so each residual block has Lipschitz constant bounded by $1 + L_{\text{GIN}} \le 2$, and the whole encoder by $2^{n_{\text{conv}}}$. This is looser than a strict 1-Lipschitz bound but is the standard practical compromise: the *upper bound* on how much two inputs can be pulled apart in feature space grows only mildly with depth, preserving distance-awareness while making the network easy to train. We also apply a `LayerNorm` after each residual GIN block; LayerNorm is not 1-Lipschitz (it rescales by the inverse running standard deviation), so it formally weakens the Lipschitz bound further, but in practice it is the single most effective stabiliser for deeper or wider GIN stacks and the trade-off is worth it.

**Why this matters for uncertainty.** This recipe is essentially the feature-extractor half of *SNGP* (Spectral-Normalized Gaussian Process; Liu et al. 2020), one of the standard "distance-aware uncertainty" methods. The intuition is that a Lipschitz-bounded encoder cannot "fold" OOD inputs into the in-distribution region of feature space &mdash; so when the ensemble members disagree on an OOD molecule, that disagreement is *forced* by the geometry of the input rather than being a lucky accident of training. In practice, spectral normalisation tends to:

- *Reduce* in-distribution test accuracy slightly (the model has less capacity to memorise the training set), and
- *Improve* the calibration of uncertainty estimates and their separation between in-distribution and OOD inputs.

**Trade-off.** The forward pass is slightly more expensive (two power-iteration steps per linear per forward), and ReLU + spectral norm together can mildly slow convergence relative to the unconstrained GELU model. In our setting that cost is small relative to ensemble training, and the resulting model is a better uncertainty estimator &mdash; which is the point.

</details>

### Graph Neural Network Model

For the ensemble members we reuse the graph neural network architecture introduced in Tutorial 3, applied to the same water solubility regression task. The same atom and bond feature encoders (`encode_atom`, `encode_bond`) and the same SMILES-to-`Data` conversion functions (`graph_from_smiles`, `data_from_graph`) are re-defined here so that this notebook is self-contained. The supported atom list defines which chemical elements the model is able to distinguish in its features &mdash; this list will become directly relevant in the OOD analysis later, where we deliberately probe the model with elements that are *not* on this list.

In [ ]:
# --- Atom and Bond Encoding ---
# Re-implements the feature encoders from Tutorial 2 / Tutorial 3 so this notebook
# stands on its own. The set of "supported_atoms" defines which chemical elements
# the GNN is able to represent as distinct one-hot features; any atom outside this
# list collapses into the single "unknown" feature, which is precisely what makes
# Tier 2 (unseen elements) a sharp distribution shift later on.

SUPPORTED_ATOMS: list[str] = ['C', 'N', 'O', 'S', 'P', 'Cl', 'Br', 'F']


def encode_atom(atom: Chem.Atom,
                supported_atoms: list[str] = SUPPORTED_ATOMS,
                ) -> np.ndarray:
    """
    Encodes an RDKit `atom` object into a fixed-size feature vector consisting of a
    one-hot atom-type indicator over `supported_atoms`, an "unknown" flag for atoms
    outside this list, and the atom's total valence, implicit valence and formal charge.

    :param atom: The Chem.Atom object to be converted.
    :param supported_atoms: A list of atom symbols that are supported for the one-hot encoding.

    :return: A numpy array of shape (n_feature, ) containing the encoded atom properties.
    """
    atom_one_hot = np.zeros(len(supported_atoms), dtype=np.float32)
    if atom.GetSymbol() in supported_atoms:
        atom_one_hot[supported_atoms.index(atom.GetSymbol())] = 1.0

    atom_unknown = float(atom.GetSymbol() not in supported_atoms)

    return np.array([
        *atom_one_hot,
        atom_unknown,
        atom.GetTotalValence(),
        atom.GetImplicitValence(),
        atom.GetFormalCharge(),
    ], dtype=np.float32)


def encode_bond(bond: Chem.Bond,
                supported_bond_types: list = [
                    Chem.BondType.SINGLE,
                    Chem.BondType.DOUBLE,
                    Chem.BondType.TRIPLE,
                ],
                ) -> np.ndarray:
    """
    Encodes an RDKit `bond` object into a fixed-size feature vector consisting of a
    one-hot bond-type indicator, an "unknown" flag, the bond's stereo descriptor and
    an aromaticity flag.
    """
    bond_type_one_hot = np.zeros(len(supported_bond_types), dtype=np.float32)
    if bond.GetBondType() in supported_bond_types:
        bond_type_one_hot[supported_bond_types.index(bond.GetBondType())] = 1.0

    bond_unknown = float(bond.GetBondType() not in supported_bond_types)

    return np.array([
        *bond_type_one_hot,
        bond_unknown,
        bond.GetStereo(),
        float(bond.GetIsAromatic()),
    ], dtype=np.float32)


# Per-atom feature dimensionality (will be augmented with graph-theoretic
# random-walk features in the next cell — the final NUM_NODE_FEATURES is defined there).
NUM_ATOM_FEATURES = len(encode_atom(Chem.Atom('C')))
NUM_EDGE_FEATURES = encode_bond(Chem.RWMol(Chem.MolFromSmiles('CC')).GetBondWithIdx(0)).shape[0]
print(f'Atom-only feature dim: {NUM_ATOM_FEATURES}, Edge feature dim: {NUM_EDGE_FEATURES}')

Atom-only feature dim: 12, Edge feature dim: 6


In addition to the chemistry-aware atom features (atom type, valence, charge, etc.), we attach a set of **graph-theoretic structural features** to each node: the diagonals of the unweighted random-walk transition matrix at powers $k = 1, 2, \ldots, 32$. The $k$-th value at node $v$ is the probability that an unweighted random walk of $k$ steps starting at $v$ returns to $v$. These features encode the local topology around each atom &mdash; for instance, an atom in a benzene ring has a characteristic return-probability pattern that distinguishes it from an atom in a long chain &mdash; and are closely related to Random Walk Structural Encodings (RWSE) used in modern graph transformers. Concatenating them with `encode_atom` raises the per-node feature dimensionality from 12 to 12 + 32 = 44.

In [ ]:
# --- SMILES -> networkx -> PyG Data pipeline, with random-walk structural features ---
# Beyond the chemistry-aware atom features from `encode_atom`, we also attach a set
# of *graph-theoretic* per-node features: the diagonals of the unweighted random-walk
# transition matrix raised to powers k = 1, 2, ..., RW_MAX_K. These return
# probabilities encode the local topology around each atom at a range of distances
# &mdash; closely related to Random Walk Structural Encodings (RWSE) used in modern
# graph transformers (e.g. GraphGPS). The concatenated node feature dimensionality
# is therefore NUM_ATOM_FEATURES + RW_MAX_K.

RW_MAX_K: int = 32   # number of random-walk steps; produces RW_MAX_K extra features per node


def random_walk_return_probs(mol: Chem.Mol, max_k: int = RW_MAX_K) -> np.ndarray:
    """
    For each atom v in `mol`, compute the probability that an unweighted random walk
    of length k starting at v returns to v, for k = 1, 2, ..., max_k.

    Walks are on the *unweighted* heavy-atom graph (no self-loops, no bond-order
    weighting): the transition matrix is P = D⁻¹ A, where A is the binary adjacency
    matrix and D the diagonal degree matrix. The k-th return probability for node v
    is the diagonal element (P^k)[v, v]. At k = 1 these are all zero (no self-loops);
    at larger k they grow as walks have time to return through different cycles, and
    therefore characterise the local ring structure around each atom.

    :param mol: RDKit `Mol` object.
    :param max_k: Maximum walk length (the smallest k is always 1).

    :return: Array of shape (n_atoms, max_k) of return probabilities.
    """
    n = mol.GetNumAtoms()
    if n == 0:
        return np.zeros((0, max_k), dtype=np.float32)

    # Unweighted, no self-loops.
    A = np.zeros((n, n), dtype=np.float64)
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        A[i, j] = 1.0
        A[j, i] = 1.0

    deg = A.sum(axis=1)
    P = np.zeros_like(A)
    nonzero = deg > 0
    P[nonzero] = A[nonzero] / deg[nonzero, None]

    out = np.zeros((n, max_k), dtype=np.float32)
    Pk = np.eye(n, dtype=np.float64)
    for k in range(max_k):
        Pk = Pk @ P
        # diag(P^{k+1}); we store k = 0..max_k-1 corresponding to walk lengths 1..max_k
        out[:, k] = np.diagonal(Pk)
    return out


def graph_from_smiles(smiles: str) -> nx.Graph:
    """
    Converts a SMILES string into a `networkx.Graph` whose nodes carry, under the
    `node_attributes` key, the per-atom feature vector defined as the concatenation
    of `encode_atom(atom)` and `random_walk_return_probs(mol)[atom_idx]`.
    Edges carry the encoded bond feature vector under the `edge_attributes` key.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Could not parse SMILES: {smiles}')

    rw_feats = random_walk_return_probs(mol, RW_MAX_K)

    graph = nx.Graph()
    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        base_feats = encode_atom(atom)
        node_feats = np.concatenate([base_feats, rw_feats[idx]]).astype(np.float32)
        graph.add_node(idx, node_attributes=node_feats)
    for bond in mol.GetBonds():
        graph.add_edge(
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx(),
            edge_attributes=encode_bond(bond),
        )
    return graph


def data_from_graph(graph: nx.Graph) -> Data:
    """
    Converts a networkx graph (as produced by `graph_from_smiles`) into a Pytorch
    Geometric `Data` object suitable for training. Optionally attaches the regression
    target `graph.graph['target']` to the resulting object under `.y`.
    """
    x = torch.tensor(np.stack([graph.nodes[n]['node_attributes'] for n in graph.nodes()]),
                     dtype=torch.float32)

    edges_src, edges_dst, edge_attrs = [], [], []
    for u, v, d in graph.edges(data=True):
        edges_src.extend([u, v])
        edges_dst.extend([v, u])
        edge_attrs.extend([d['edge_attributes'], d['edge_attributes']])

    edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
    edge_attr = torch.tensor(np.stack(edge_attrs), dtype=torch.float32) if edge_attrs else \
                torch.zeros((0, NUM_EDGE_FEATURES), dtype=torch.float32)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if 'target' in graph.graph:
        data.y = torch.tensor([graph.graph['target']], dtype=torch.float32)
    return data


# --- final per-node feature dimensionality (atom features + RW return probabilities) ---
NUM_NODE_FEATURES = NUM_ATOM_FEATURES + RW_MAX_K
assert NUM_NODE_FEATURES == data_from_graph(graph_from_smiles('CCO')).x.shape[1]
print(f'Node feature dim (atom + RW): {NUM_NODE_FEATURES} '
      f'(= {NUM_ATOM_FEATURES} atom + {RW_MAX_K} random-walk)')

# --- example: show random-walk features for a small molecule with a 5-ring ---
example_smiles = 'c1ccc2ccccc2c1'  # naphthalene
example_graph = graph_from_smiles(example_smiles)
example_data = data_from_graph(example_graph)
print(f'\nExample SMILES: {example_smiles}')
print(f'  num_nodes={example_data.x.shape[0]}, node feature dim={example_data.x.shape[1]}')
rw_part = example_data.x[0, NUM_ATOM_FEATURES:NUM_ATOM_FEATURES + 8].numpy()
print(f'  random-walk return probs (atom 0, k=1..8): {rw_part}')

Node feature dim (atom + RW): 44 (= 12 atom + 32 random-walk)

Example SMILES: c1ccc2ccccc2c1
  num_nodes=10, node feature dim=44
  random-walk return probs (atom 0, k=1..8): [0.         0.5        0.         0.35416666 0.         0.28761575
 0.         0.24945344]


In [ ]:
# --- Graph neural network for solubility regression with mean-variance estimation ---
# This is a *residual* GIN-based encoder. Concretely:
#
#   1. A single spectral-normalised linear layer projects the raw node features
#      (atom one-hot + valence/charge + 32 random-walk features) into a hidden
#      space of dimension `hidden_dim`. From here on every layer operates in this
#      same hidden space.
#   2. Each subsequent GINConv layer produces a *residual update* of the current
#      node embedding:  x ← x + ReLU( GINConv(x) ). The conv layer never replaces
#      the running embedding; it only adds a correction to it. This is the same
#      trick that makes very deep ResNets / Graph Transformers trainable, and it
#      makes the network easier to optimise without sacrificing expressiveness.
#   3. Every nn.Linear in the network is spectral-normalised. With ReLU
#      activations and pure-residual updates, the per-block Lipschitz constant
#      is bounded by 1 + L_conv (≤ 2), and the whole encoder has Lipschitz
#      constant at most 2^n_conv. This is looser than the strict 1-Lipschitz
#      bound of the non-residual version but still keeps the network
#      "distance-aware" in the SNGP sense for the OOD-detection signal.
#   4. Each residual block is followed by a LayerNorm on the hidden-dim axis.
#      LayerNorm is *not* 1-Lipschitz (it rescales by 1/std), so it formally
#      weakens the Lipschitz bound, but it dramatically improves training
#      stability for deeper / wider GIN stacks, which is the dominant
#      practical concern at this depth.
#   5. The GINConv aggregation uses train_eps=False: the (1+ε) self-loop scale
#      is held fixed at its default rather than learned. With residual
#      connections and LayerNorm in place, a learnable ε no longer adds useful
#      capacity and only contributes one more axis along which the spectral-norm
#      Lipschitz argument is harder to reason about.
#
# This is conceptually similar to the architecture used in distance-aware
# uncertainty methods such as SNGP (Liu et al. 2020) with residual connections,
# which is the configuration most commonly used in production.

class Exponential(nn.Module):
    """
    Exponential activation as a Module. Forces the variance head output to be strictly
    positive. The `+ 0` is an autograd workaround for a subtle PyTorch issue that has
    bitten the reference implementation in `truthful_counterfactuals` &mdash; without it
    some PyTorch versions silently zero out gradients through the exp.
    """

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.exp(x) + 0


def sn_linear(in_features: int, out_features: int) -> nn.Module:
    """
    Spectral-normalised linear layer: wraps `nn.Linear` with the modern parametrisation
    API from `torch.nn.utils.parametrizations`. Bounds the layer's spectral norm
    (largest singular value of W) to ≤ 1, which means the linear map itself is
    1-Lipschitz.
    """
    return spectral_norm(nn.Linear(in_features, out_features))


def make_gin_mlp(in_dim: int, hidden_dim: int) -> nn.Module:
    """
    The 2-layer ReLU MLP that we hand to each `GINConv`. Both linear layers are
    spectral-normalised so that the entire message-passing transform is Lipschitz-bounded.
    """
    return nn.Sequential(
        sn_linear(in_dim, hidden_dim),
        nn.ReLU(),
        sn_linear(hidden_dim, hidden_dim),
    )


class SolubilityGNN(pl.LightningModule):
    """
    Residual GIN-based graph neural network for solubility regression with mean-variance
    estimation, spectral-normalised linear layers, and ReLU activations.

    The input features are projected once into a hidden space; every subsequent GIN
    layer only contributes a residual update. All nn.Linear layers carry spectral
    normalisation.

    :param node_dim: Number of input node features (set by `encode_atom` + RW features).
    :param hidden_dim: Width of the hidden node embeddings.
    :param n_conv_layers: Number of residual GINConv layers.
    :param learning_rate: Adam learning rate.
    :param mve_beta: β coefficient of the β-NLL re-weighting.
    :param eps: Numerical floor on σ² to prevent log(0)/1/σ² blow-ups.
    """

    def __init__(self,
                 node_dim: int = NUM_NODE_FEATURES,
                 hidden_dim: int = 64,
                 n_conv_layers: int = 3,
                 learning_rate: float = 1e-3,
                 mve_beta: float = 0.5,
                 eps: float = 1e-6,
                 ) -> None:
        super().__init__()
        self.save_hyperparameters()

        # --- single input projection: raw node features -> hidden_dim ---
        # All subsequent layers operate in this hidden space and only add residuals.
        self.input_proj = sn_linear(node_dim, hidden_dim)

        # --- residual GIN encoder: every layer is hidden_dim -> hidden_dim ---
        self.convs = nn.ModuleList([
            GINConv(make_gin_mlp(hidden_dim, hidden_dim), train_eps=False)
            for _ in range(n_conv_layers)
        ])

        # LayerNorms applied after each residual update, one per conv block.
        # LayerNorm normalises across the hidden feature dim per-node, which
        # stabilises training of deeper / wider GNN stacks and keeps the running
        # node embedding from drifting in magnitude as residuals accumulate.
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(n_conv_layers)
        ])

        # --- mean head μ(x) ---
        self.head = nn.Sequential(
            sn_linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            sn_linear(hidden_dim, 1),
        )

        # --- variance head σ²(x) ---
        # hidden_dim -> 32 -> 1, Exponential activation so σ² > 0.
        self.variance_head = nn.Sequential(
            sn_linear(hidden_dim, 32),
            nn.ReLU(),
            sn_linear(32, 1),
            Exponential(),
        )

    # --- forward passes ---

    def _graph_embedding(self, data) -> torch.Tensor:
        x, edge_index, batch = data.x, data.edge_index, data.batch
        # Single projection to hidden space
        x = self.input_proj(x)
        # Pure residual updates from here on, each followed by LayerNorm.
        for conv, norm in zip(self.convs, self.norms):
            x = norm(x + F.relu(conv(x, edge_index)))
        return global_add_pool(x, batch)

    def forward(self, data) -> torch.Tensor:
        """Mean prediction only. Used at inference when σ² is not needed."""
        return self.head(self._graph_embedding(data)).squeeze(-1)

    def forward_with_variance(self, data) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns `(μ, σ²)` for the input batch."""
        h = self._graph_embedding(data)
        mu = self.head(h).squeeze(-1)
        var = self.variance_head(h).squeeze(-1) + self.hparams.eps
        return mu, var

    # --- β-NLL loss ---

    def mve_beta_nll(self,
                     y_true: torch.Tensor,
                     y_pred: torch.Tensor,
                     var: torch.Tensor,
                     ) -> torch.Tensor:
        """
        β-NLL loss of Seitzer et al. (2022). The `var.detach()**β` factor re-weights
        the standard Gaussian NLL so that high-variance samples contribute more than
        they would under vanilla NLL. Setting `β = 0` recovers vanilla NLL.
        """
        var = var + self.hparams.eps
        std = torch.sqrt(var)
        per_sample = 0.5 * std.detach() ** (2 * self.hparams.mve_beta) * (
            torch.log(var) + (y_true - y_pred) ** 2 / var
        )
        return per_sample.mean()

    # --- Lightning hooks ---

    def training_step(self, batch, batch_idx):
        y_hat, var = self.forward_with_variance(batch)
        loss = self.mve_beta_nll(batch.y, y_hat, var)
        self.log('train_nll', loss, on_epoch=True, prog_bar=True, batch_size=batch.num_graphs)
        return loss

    def validation_step(self, batch, batch_idx):
        y_hat = self.forward(batch)
        loss = F.mse_loss(y_hat, batch.y)
        self.log('val_mse', loss, on_epoch=True, prog_bar=True, batch_size=batch.num_graphs)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

    def predict_smiles(self, smiles: str) -> float:
        """Convenience method: run a forward pass on a single SMILES string."""
        self.eval()
        with torch.no_grad():
            data = data_from_graph(graph_from_smiles(smiles))
            data.batch = torch.zeros(data.x.shape[0], dtype=torch.long)
            return float(self.forward(data).cpu().item())


### Training the Ensemble

We now train the ensemble. The hyperparameters that govern its construction are exposed at the top of the next cell. The training procedure exercises five independent diversity axes per member (seed, data subsample, batch size, learning rate, architecture), with β-NLL applied from epoch 0 and persistent gradient clipping for stability.

- `ENSEMBLE_SIZE` &mdash; the number of independent GNN members. Defaults to $N = 5$.
- `SUBSAMPLE_FRAC` &mdash; the fraction of the training set used to train each member. Defaults to **0.6** (lower than the usual 0.8) to emphasise data-subset diversity.
- `EPOCH_COUNT` &mdash; **50** epochs per member.
- `BATCH_SIZE_MEAN` / `BATCH_SIZE_JITTER` &mdash; each member draws its own batch size uniformly from `[mean - jitter, mean + jitter]`.
- `LR_MEAN` / `LR_LOG_JITTER` &mdash; each member also gets its own learning rate, drawn log-uniformly within a factor of $10^{\pm \text{LR\_LOG\_JITTER}}$ of the mean (here ~3× either direction).
- `HIDDEN_DIM_CHOICES`, `N_CONV_CHOICES` &mdash; each member draws its own architecture from this set: a hidden width in `{48, 64, 80, 96, 112}` and a depth in `{2, 3, 4}` GCN layers.
- `GRAD_CLIP_VAL` &mdash; persistent by-value gradient clipping at 50.

**📝 Note.** With five ensemble members and 50 epochs each, training the full ensemble takes a few minutes on CPU and is noticeably faster on a GPU. Larger ensembles linearly increase this cost.

In [ ]:
# --- Ensemble training configuration ---

ENSEMBLE_SIZE: int = 5       # Number of GNN members in the deep ensemble.
EPOCH_COUNT: int = 75        # Training epochs per member.
SUBSAMPLE_FRAC: float = 0.8  # Fraction of the training set used per member (lower => more diversity).

# Optimisation-hyperparameter randomisation: each ensemble member draws a slightly
# different batch size and learning rate around these means.
BATCH_SIZE_MEAN: int = 32
BATCH_SIZE_JITTER: int = 24       # uniform integer jitter in [-jitter, +jitter]
LR_MEAN: float = 1e-3
LR_LOG_JITTER: float = 0.5        # multiplicative jitter ~ 10**uniform(-x, +x)  (~3×)

# Architectural randomisation: each member uses a slightly different GNN architecture.
# This is the strongest single axis of diversity beyond the data subsample.
HIDDEN_DIM_CHOICES: list[int] = [48, 64, 80, 96, 112]
N_CONV_CHOICES: list[int] = [2, 3, 4]

# β-NLL training is somewhat sensitive to gradient spikes, so we keep gradient clipping
# active throughout training (rather than only after a warmup phase).
GRAD_CLIP_VAL: float = 50.0


# --- train-test split ---
# The same 80/20 split is shared across all ensemble members; only the within-train
# subsample changes between members. This way the test set remains untouched by any
# member and provides an unbiased estimate of ensemble performance.

indices = list(range(len(data_frame)))
random.shuffle(indices)
n_test = int(len(indices) * 0.2)
test_indices = indices[:n_test]
train_indices = indices[n_test:]
print(f'Train set size: {len(train_indices)}, test set size: {len(test_indices)}')

# --- build PyG Data objects once ---
print('Converting molecules to PyG Data objects...')
data_list: list[Data] = []
for i, row in data_frame.iterrows():
    graph = graph_from_smiles(row['smiles'])
    graph.graph['target'] = row['solubility']
    data_list.append(data_from_graph(graph))
print(f'Built {len(data_list)} Data objects')


def sample_member_hyperparams(rng: random.Random) -> dict:
    """
    Draw a fresh set of training and architectural hyperparameters for a single
    ensemble member. Each member gets its own seed, batch size, learning rate,
    hidden dimensionality and number of GCN layers.
    """
    seed = rng.randint(0, 2**31 - 1)
    batch_size = max(8, BATCH_SIZE_MEAN + rng.randint(-BATCH_SIZE_JITTER, BATCH_SIZE_JITTER))
    lr = LR_MEAN * (10 ** rng.uniform(-LR_LOG_JITTER, LR_LOG_JITTER))
    hidden_dim = rng.choice(HIDDEN_DIM_CHOICES)
    n_conv = rng.choice(N_CONV_CHOICES)
    return {
        'seed': seed,
        'batch_size': batch_size,
        'learning_rate': lr,
        'hidden_dim': hidden_dim,
        'n_conv_layers': n_conv,
    }


def train_ensemble_member(member_idx: int,
                          train_indices_subset: list[int],
                          test_indices: list[int],
                          hparams: dict,
                          epochs: int = EPOCH_COUNT,
                          ) -> SolubilityGNN:
    """
    Train a single ensemble member directly with the β-NLL loss for `epochs` epochs.

    A fresh `SolubilityGNN` is instantiated with the member-specific architecture and
    learning rate. The supplied `seed` is used to seed both PyTorch (for the random
    weight initialisation) and Lightning, so that members differ in a controlled,
    reproducible way. Gradient clipping (by value, threshold `GRAD_CLIP_VAL`) is on
    throughout training to suppress the rare large-gradient events characteristic of
    NLL-based losses.
    """
    pl.seed_everything(hparams['seed'], workers=True)

    train_loader = DataLoader(
        [data_list[i] for i in train_indices_subset],
        batch_size=hparams['batch_size'], shuffle=True,
    )
    test_loader = DataLoader(
        [data_list[i] for i in test_indices],
        batch_size=hparams['batch_size'], shuffle=False,
    )

    model = SolubilityGNN(
        hidden_dim=hparams['hidden_dim'],
        n_conv_layers=hparams['n_conv_layers'],
        learning_rate=hparams['learning_rate'],
    )

    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator='auto',
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
        gradient_clip_val=GRAD_CLIP_VAL,
        gradient_clip_algorithm='value',
    )
    print(f'--- Training ensemble member {member_idx + 1}/{ENSEMBLE_SIZE} '
          f'(seed={hparams["seed"]}, bs={hparams["batch_size"]}, '
          f'lr={hparams["learning_rate"]:.4f}, hidden={hparams["hidden_dim"]}, '
          f'n_conv={hparams["n_conv_layers"]}) on {len(train_indices_subset)} samples ---')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)

    model.eval()
    return model


# --- run the training for all ensemble members ---

meta_rng = random.Random(2026)

ensemble: list[SolubilityGNN] = []
member_train_indices: list[list[int]] = []
member_hparams: list[dict] = []

for member_idx in range(ENSEMBLE_SIZE):
    k = int(len(train_indices) * SUBSAMPLE_FRAC)
    subset = meta_rng.sample(train_indices, k=k)
    member_train_indices.append(subset)

    hparams = sample_member_hyperparams(meta_rng)
    member_hparams.append(hparams)

    model = train_ensemble_member(member_idx, subset, test_indices, hparams)
    ensemble.append(model)

print(f'\nTrained ensemble of {len(ensemble)} members.')
print('Per-member hyperparameters:')
for i, h in enumerate(member_hparams):
    print(f'  member {i+1}: seed={h["seed"]}, bs={h["batch_size"]}, '
          f'lr={h["learning_rate"]:.4f}, hidden={h["hidden_dim"]}, '
          f'n_conv={h["n_conv_layers"]}')


Train set size: 8818, test set size: 2204
Converting molecules to PyG Data objects...


Seed set to 238388027
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Built 11022 Data objects
--- Training ensemble member 1/5 (seed=238388027, bs=18, lr=0.0019, hidden=112, n_conv=3) on 7054 samples ---



Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

<div style="background: #fff0c2; padding: 10px; border-style: solid; border-width: 1px; border-color: #dbad21; border-radius: 3px; color: black;">

**🛠️ Exercise 4.1** $\cdot$ Re-run the training above with `ENSEMBLE_SIZE = 5` (or larger) and observe how the uncertainty estimates downstream change. Do the in-distribution calibration plots become smoother? Does the separation between OOD tiers become clearer?

</div>

### Computing Uncertainty

With the trained ensemble in hand, computing predictions and their associated uncertainties is straightforward: for a given input molecule we run a forward pass through every member, collect the $N$ resulting scalar predictions, and report their mean and standard deviation. The mean serves as the final ensemble prediction; the standard deviation is the uncertainty estimate we will study in the rest of this notebook.

**📝 Note.** We use the *unbiased* standard deviation (`ddof=1`) because we are treating the $N$ predictions as a small sample drawn from an implicit distribution of "models we could have trained". For very small ensembles ($N = 2$ or $N = 3$) the resulting estimate is noisy, but its rank-order &mdash; which inputs are *more* uncertain than which others &mdash; is already informative.

In [ ]:
# --- Ensemble prediction with moment-matched uncertainty ---
# Aggregates the per-member Gaussians (μ_m, σ²_m) into a single Gaussian (μ_*, σ²_*)
# via the law of total variance. The total variance decomposes additively into an
# aleatoric component (mean of per-member σ²) and an epistemic component (variance
# of per-member μ across the ensemble) &mdash; both useful diagnostics on their own.

@torch.no_grad()
def predict_with_uncertainty(smiles_list: list[str],
                              ensemble: list[SolubilityGNN],
                              ) -> dict:
    """
    Run every ensemble member on every SMILES string and return moment-matched
    ensemble statistics.

    For each input molecule we collect per-member predicted means and variances,
    then combine them via:

        μ*(x)  = (1/M) Σ μ_m(x)
        σ²*(x) = (1/M) Σ (σ²_m(x) + μ²_m(x)) − μ²*(x)
                ≡ aleatoric(x) + epistemic(x)

    where `aleatoric` is the mean of the per-member variances and `epistemic` is
    the population variance of the per-member means.

    :param smiles_list: List of SMILES strings to predict on.
    :param ensemble: List of trained `SolubilityGNN` models.

    :return: Dict with keys:
        `mus` (shape (N, M)), `vars` (shape (N, M)),
        `mean` (shape (N,)), `std` (shape (N,)),
        `aleatoric_std` (shape (N,)), `epistemic_std` (shape (N,)).
    """
    N, M = len(smiles_list), len(ensemble)
    mus = np.full((N, M), np.nan, dtype=np.float32)
    vars_ = np.full((N, M), np.nan, dtype=np.float32)

    for i, smi in enumerate(smiles_list):
        try:
            graph = graph_from_smiles(smi)
        except Exception:
            continue
        data = data_from_graph(graph)
        data.batch = torch.zeros(data.x.shape[0], dtype=torch.long)
        for j, model in enumerate(ensemble):
            mu, var = model.forward_with_variance(data)
            mus[i, j] = float(mu.cpu().item())
            vars_[i, j] = float(var.cpu().item())

    # μ* — ensemble mean
    mean_pred = np.nanmean(mus, axis=1)
    # Aleatoric: (1/M) Σ σ²_m  — average within-member variance.
    aleatoric_var = np.nanmean(vars_, axis=1)
    # Epistemic: population variance of the per-member means (ddof=0 to match the
    # law-of-total-variance derivation, which divides by M and not by M-1).
    epistemic_var = np.nanmean(mus**2, axis=1) - mean_pred**2
    # Clip rare tiny-negative epistemic values arising from floating-point noise.
    epistemic_var = np.clip(epistemic_var, a_min=0.0, a_max=None)

    total_var = aleatoric_var + epistemic_var
    total_std = np.sqrt(total_var)

    return {
        'mus': mus,
        'vars': vars_,
        'mean': mean_pred,
        'std': total_std,
        'aleatoric_std': np.sqrt(aleatoric_var),
        'epistemic_std': np.sqrt(epistemic_var),
    }


# --- example usage ---
example_smiles_list = ['CCO', 'c1ccccc1', 'CC(=O)O']
example_result = predict_with_uncertainty(example_smiles_list, ensemble)
print('Mean predictions :', example_result['mean'])
print('Total std        :', example_result['std'])
print('Aleatoric std    :', example_result['aleatoric_std'])
print('Epistemic std    :', example_result['epistemic_std'])

## **4.3** $\cdot$ 📚 Demonstrating Uncertainty Quality

Producing an uncertainty estimate is only half the work &mdash; we also need to convince ourselves that the estimate actually *means* something. A useful uncertainty score should satisfy two complementary properties:

1. **Calibration on in-distribution data.** When the model is operating on inputs that resemble its training data, the uncertainty estimate should correlate with the actual prediction error. Molecules predicted with high uncertainty should, on average, also have larger absolute errors than molecules predicted with low uncertainty.

2. **Sensitivity to distribution shift.** When the model is presented with inputs progressively further from its training distribution, the uncertainty estimate should rise correspondingly &mdash; even (especially) when ground-truth labels are not available for these shifted inputs.

The remainder of this section evaluates the ensemble on both criteria.

### In-Distribution Calibration

We first evaluate the ensemble on the held-out 20% test set drawn from the same AqSolDB distribution as the training data. Because we have ground-truth solubility values for these molecules, we can compute the absolute error of every prediction and directly check whether the ensemble's uncertainty estimate correlates with this error.

In [ ]:
# --- Predict on the in-distribution test set ---

test_smiles = data_frame['smiles'].iloc[test_indices].tolist()
test_targets = data_frame['solubility'].iloc[test_indices].to_numpy()

test_result = predict_with_uncertainty(test_smiles, ensemble)
test_mean = test_result['mean']
test_std = test_result['std']
test_aleatoric = test_result['aleatoric_std']
test_epistemic = test_result['epistemic_std']
test_abs_error = np.abs(test_mean - test_targets)

# Aggregate statistics:
print(f'Ensemble test RMSE      : {np.sqrt(np.mean((test_mean - test_targets)**2)):.3f} logS')
print(f'Ensemble test MAE       : {np.mean(test_abs_error):.3f} logS')
print(f'Pearson(|error|, total) : {np.corrcoef(test_abs_error, test_std)[0, 1]:.3f}')
print(f'Pearson(|error|, aleat.): {np.corrcoef(test_abs_error, test_aleatoric)[0, 1]:.3f}')
print(f'Pearson(|error|, epist.): {np.corrcoef(test_abs_error, test_epistemic)[0, 1]:.3f}')


# --- Scatter |error| vs total uncertainty, with a binned running mean ---

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, vals, name in zip(
    axes,
    [test_std, test_aleatoric, test_epistemic],
    ['Total σ_*', 'Aleatoric (within-network)', 'Epistemic (between-network)'],
):
    ax.scatter(vals, test_abs_error, alpha=0.35, s=14, c='steelblue',
               edgecolor='none', label='test molecule')

    n_bins = 10
    bin_edges = np.linspace(vals.min(), vals.max(), n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_means = np.array([
        test_abs_error[(vals >= lo) & (vals < hi)].mean()
        if ((vals >= lo) & (vals < hi)).any() else np.nan
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:])
    ])
    ax.plot(bin_centers, bin_means, color='crimson', marker='o', linewidth=2,
            label='binned mean |error|')
    ax.set_xlabel(f'{name}')
    ax.grid(alpha=0.3)
    ax.set_title(name)
    ax.legend(loc='upper left', fontsize=9)

axes[0].set_ylabel('Absolute prediction error |y − ŷ| [logS]')
fig.suptitle(f'In-distribution calibration  (N = {ENSEMBLE_SIZE} ensemble members, MVE)',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

**📊 Calibration Scatters.** The three panels decompose the ensemble's uncertainty signal into its components. On each panel the horizontal axis is one of the three uncertainty estimates &mdash; *total* $\sigma_*$ (left), *aleatoric* contribution from the per-member variance heads (middle), and *epistemic* contribution from the disagreement between the per-member means (right) &mdash; and the vertical axis is the absolute prediction error of the ensemble mean on the in-distribution test set. The crimson line shows the average absolute error within uncertainty bins. A well-behaved MVE ensemble should produce a positive trend on the *total* and *epistemic* panels (uncertainty rising with error), and a less monotone but still informative signal on the *aleatoric* panel &mdash; the aleatoric component captures *intrinsic* label noise rather than model misspecification, so its correlation with |error| reflects whichever molecules the model has implicitly identified as harder to predict.

The reported Pearson correlations quantify each panel's trend in a single number. Values noticeably above zero indicate that the corresponding uncertainty component carries information about the ensemble's own reliability. Comparing the three correlations is itself informative: an MVE ensemble in which only the *epistemic* correlation is meaningful is essentially a vanilla deep ensemble &mdash; the variance heads contributed nothing; one in which only the *aleatoric* correlation is meaningful but the epistemic one is flat is a sign that the ensemble members converged to nearly identical solutions and the deep-ensemble half of the recipe failed.

In [ ]:
# --- Parity plot coloured by total uncertainty ---

fig, ax = plt.subplots(figsize=(7, 5))

# Sort so that high-uncertainty points are drawn on top and remain visible.
order = np.argsort(test_std)
sc = ax.scatter(test_targets[order], test_mean[order],
                c=test_std[order], cmap='viridis', s=20, alpha=0.75, edgecolor='none')

# y = x reference line
lo, hi = min(test_targets.min(), test_mean.min()), max(test_targets.max(), test_mean.max())
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, alpha=0.6, label='y = x')

cb = fig.colorbar(sc, ax=ax)
cb.set_label('Total uncertainty σ_* (MVE + ensemble)')

ax.set_xlabel('True solubility [logS]')
ax.set_ylabel('Ensemble mean prediction [logS]')
ax.set_title('Parity plot (in-distribution), coloured by uncertainty')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**📊 Parity Plot.** The horizontal axis shows the true measured solubility, the vertical axis the ensemble's mean prediction, and each point's colour encodes the ensemble's uncertainty estimate. The dashed black line is the ideal $y = x$ relationship. A well-calibrated ensemble should show two patterns simultaneously: (1) the cloud of points lies tightly along the diagonal, indicating low prediction error on average; and (2) the *yellow / high-uncertainty* points are precisely the ones that stray furthest from the diagonal, while the *dark / low-uncertainty* points cluster tightly around it. If high-uncertainty points were randomly scattered &mdash; some on the diagonal, some far from it &mdash; the uncertainty estimate would be uninformative.

<div style="background: #fff0c2; padding: 10px; border-style: solid; border-width: 1px; border-color: #dbad21; border-radius: 3px; color: black;">

**🛠️ Exercise 4.2** $\cdot$ Re-train the ensemble with `SUBSAMPLE_FRAC = 1.0` &mdash; i.e. every member sees the *same* 100% of the training data, with the only remaining source of diversity being random weight initialisation and batch ordering. How does the Pearson correlation between uncertainty and error in the calibration scatter change? What does this tell you about how much of the ensemble's signal comes from data subsampling versus initialisation noise?

</div>

### Out-of-Distribution Detection

A calibrated in-distribution uncertainty score is necessary but not sufficient for a UQ method to be useful in practice. What we ultimately care about is whether the ensemble's uncertainty *rises* when the model is presented with molecules outside the distribution it was trained on &mdash; ideally well before the model is asked to make predictions that downstream users would otherwise blindly trust.

To probe this, we construct three out-of-distribution sets, each ~50 molecules in size, ordered by how far they sit from the AqSolDB training distribution:

| Tier | Description | Source |
|------|-------------|--------|
| **0 (In-dist.)** | Held-out 20% of AqSolDB | The same test set used above |
| **1 (Near-OOD)** | Heaviest molecules in AqSolDB | Programmatic filter on AqSolDB by molecular weight |
| **2 (Mid-OOD)** | Unseen chemical elements | Hand-curated SMILES with Si / Se / B / Ge / Sn / As / Te / I |
| **3 (Far-OOD)** | Large polycyclic aromatic hydrocarbons (PAHs) | `load_dataset_pahs` &mdash; ~50 PAHs spanning 10–42 heavy atoms, from naphthalene up to decacene |

The expectation is that ensemble uncertainty should rise monotonically across these tiers. Note that for tiers 2 and 3 we do *not* have ground-truth solubility values &mdash; which is exactly the realistic deployment scenario UQ is designed for: knowing whether to trust a prediction *before* you can validate it experimentally.

In [ ]:
# --- Tier 1: Near-OOD: the 50 heaviest molecules in AqSolDB ---
# These molecules are *inside* the AqSolDB distribution by definition (since they are
# AqSolDB rows), but they sit in its size tail. The training subsamples have likely
# seen only a handful of comparably large molecules each, so the ensemble's behaviour
# here is a mild stress test.

from rdkit.Chem import Descriptors

tier1_pool = data_frame.copy()
tier1_pool['mw'] = tier1_pool['mol'].apply(lambda m: Descriptors.MolWt(m))
tier1_df = tier1_pool.nlargest(50, 'mw').reset_index(drop=True)
tier1_smiles = tier1_df['smiles'].tolist()
print(f'Tier 1 set: {len(tier1_smiles)} molecules, MW range '
      f'{tier1_df["mw"].min():.0f} - {tier1_df["mw"].max():.0f} Da')

**📝 Note.** Tier 1 molecules are still drawn from AqSolDB itself, so the ensemble *could* in principle have seen some of them during training. We retain them in the analysis as a deliberately mild OOD probe: even within an "in-distribution" dataset, the tails of the size distribution receive sparser training signal than the bulk, and the uncertainty estimate should reflect that. If the ensemble's uncertainty does *not* rise here, that is already an early warning that the UQ method is too blunt.

In [ ]:
# --- Tier 2: Mid-OOD: unseen chemical elements (~50 SMILES) ---
# Each of these molecules contains at least one atom *not* in supported_atoms.
# The atom encoder collapses every such atom into a single "unknown" feature,
# discarding the chemistry-specific information entirely. The ensemble has
# therefore literally never seen these elements as anything other than a
# generic "unknown" placeholder during training.

tier3_smiles = ['C[Si](C)(C)C', 'CC[Si](C)(C)C', 'C[Si](C)(C)O', 'C[Si](C)(C)OC', 'C[Si](C)(C)c1ccccc1', 'C[Si](C)(C)O[Si](C)(C)C', 'C[Si](C)(C)OC(C)(C)C', '[Si](C)(C)(C)C(=O)O', 'C[Se]C', 'CC[Se]CC', 'C[Se]c1ccccc1', 'c1ccc2[se]ccc2c1', 'C[Se][Se]C', '[Se]=C(N)N', 'C[As](C)C', 'C[As](=O)(O)O', 'C[As](C)(C)=O', '[As](=O)(O)(O)O', 'OB(O)c1ccccc1', 'OB(O)CC', 'OB(O)CCO', 'OB(O)c1ccc(O)cc1', 'B(OC)(OC)OC', 'OB1OB(O)OB(O)O1', 'C[Ge](C)(C)C', 'C[Ge](C)(C)CC', 'C[Ge](C)(C)O', 'C[Sn](C)(C)C', 'CC[Sn](CC)(CC)CC', 'C[Sn](C)(C)O', 'C[Te]C', 'CC[Te]CC', 'CI', 'CCI', 'Ic1ccccc1', 'Ic1ccc(I)cc1', 'CC(=O)CI', 'CCCCCCCCI', 'FC(F)(F)I', 'C[Si](C)(C)CCN', 'C[Se]CC(=O)O', 'OB(O)c1ccc(C(=O)O)cc1', '[Sn](C)(C)(C)Cl', 'C[Si](C)(C)CC=C', 'OC[Si](C)(C)C', 'C[Se]CC', '[As](C)(C)c1ccccc1', 'OB(O)CCCC', 'C[Ge](C)(C)c1ccccc1', 'ICCC(=O)O']
print(f'Tier 2 set: {len(tier3_smiles)} molecules')

**🔍 Why this set is OOD.** The information loss at the feature encoder is severe: a silicon atom and a tellurium atom both reduce to the *same* "unknown" one-hot bit. The ensemble has no way to assign meaningful electronic or steric properties to these atoms, and the prediction from any individual member is essentially driven by the surrounding (familiar) atoms plus a generic placeholder. Different members, biased by different 80% subsamples, will resolve this ambiguity differently &mdash; producing high ensemble disagreement.

In [ ]:
# --- Tier 3: Far-OOD: large polycyclic aromatic hydrocarbons (PAHs) (~50 SMILES) ---
# Loaded from `xai_chem_review/datasets/pahs.csv` via the loader function
# `load_dataset_pahs`. These molecules consist purely of fused aromatic carbon rings
# (with a few methylated variants for diversity), ranging from naphthalene (10 atoms)
# up to decacene (42 atoms) and including larger peri-fused systems like coronene.
# They represent a chemistry essentially absent from the AqSolDB training distribution:
# pure C/H composition (so atom-feature one-hots collapse to a single channel),
# hexagonal fused-aromatic topology, very rigid, very large, and famously poorly soluble.

pahs_df = load_dataset_pahs()
tier4_smiles = pahs_df['smiles'].tolist()
print(f'Tier 3 set: {len(tier4_smiles)} PAH molecules')
print(f'  Heavy-atom range: {pahs_df["n_heavy_atoms"].min()} - {pahs_df["n_heavy_atoms"].max()}')
print(f'  MW range:         {pahs_df["mol_weight"].min():.0f} - {pahs_df["mol_weight"].max():.0f} Da')
pahs_df.head()

**🔍 Why this set is OOD.** PAHs differ from AqSolDB along nearly every axis simultaneously. (1) *Composition:* they contain only carbon and (implicit) hydrogen, so the atom-feature one-hot encoding collapses to a single channel and the model never sees the heteroatom-pattern signal it relies on for ordinary drug-like molecules. (2) *Topology:* fused hexagonal ring systems are vastly larger and more rigid than the small ring fragments typical of AqSolDB; the very large entries here (8–10 fused rings) have heavy-atom counts well into the tail of AqSolDB. (3) *Solubility:* large PAHs are famously among the least water-soluble organic molecules known, with experimental logS values often below −10 &mdash; well outside the predictive range of any model trained on a drug-like distribution. The ensemble's pooled graph embedding has effectively no training precedent for inputs that combine all three of these properties at once.

In [ ]:
# --- Compute ensemble uncertainties for every tier ---

print('Predicting on all three OOD tiers...')
tier0_res = predict_with_uncertainty(test_smiles, ensemble)        # in-distribution baseline
tier1_res = predict_with_uncertainty(tier1_smiles, ensemble)
tier2_res = predict_with_uncertainty(tier3_smiles, ensemble)       # unseen element (variable name kept)
tier3_res = predict_with_uncertainty(tier4_smiles, ensemble)       # PAHs (variable name kept)

# Drop NaNs that arise from SMILES that the encoder could not handle.
def _clean(arr):
    return arr[~np.isnan(arr)]

tier_total = [_clean(r['std']) for r in
              (tier0_res, tier1_res, tier2_res, tier3_res)]
tier_aleat = [_clean(r['aleatoric_std']) for r in
              (tier0_res, tier1_res, tier2_res, tier3_res)]
tier_epist = [_clean(r['epistemic_std']) for r in
              (tier0_res, tier1_res, tier2_res, tier3_res)]
tier_labels = ['Tier 0\n(in-dist.)', 'Tier 1\n(heavy)',
               'Tier 2\n(unseen\nelement)', 'Tier 3\n(PAHs)']

# Print a numerical summary alongside the figure.
print('\nPer-tier statistics (median):')
print(f'  {"tier":30s}  {"total":>8s}  {"aleat.":>8s}  {"epist.":>8s}  {"n":>4s}')
for label, t, a, e in zip(tier_labels, tier_total, tier_aleat, tier_epist):
    label_inline = label.replace("\n", " ")
    print(f'  {label_inline:30s}  {np.median(t):8.3f}  {np.median(a):8.3f}  '
          f'{np.median(e):8.3f}  {len(t):4d}')


# --- Three-panel violin plot: total / aleatoric / epistemic ---

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True)
cmap = plt.cm.plasma

for ax, data, name in zip(
    axes,
    [tier_total, tier_aleat, tier_epist],
    ['Total σ_*', 'Aleatoric (mean σ_m)', 'Epistemic (std of μ_m)'],
):
    parts = ax.violinplot(data, positions=range(len(data)),
                          showmedians=True, widths=0.85)
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(cmap(i / (len(data) - 1)))
        body.set_alpha(0.75)
        body.set_edgecolor('black')
    for i, vals in enumerate(data):
        jitter = np.random.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(np.full_like(vals, i) + jitter, vals, s=6,
                   color='black', alpha=0.25)
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels(tier_labels, fontsize=9)
    ax.set_ylabel(name)
    ax.set_title(name)
    ax.grid(axis='y', alpha=0.3)
    # Clip y-axis to suppress extreme outliers that would otherwise dominate the scale.
    # We pad slightly above the 98th percentile so the bulk of each distribution
    # remains visible without being squashed by a handful of tail points.
    all_vals = np.concatenate([v for v in data if len(v) > 0])
    if len(all_vals) > 0:
        y_lo = max(0.0, np.percentile(all_vals, 1))
        y_hi = np.percentile(all_vals, 98)
        pad = 0.05 * (y_hi - y_lo + 1e-6)
        ax.set_ylim(y_lo - pad, y_hi + pad)

fig.suptitle('Ensemble uncertainty rises with distribution-shift severity '
             '(MVE + moment matching)', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

**📊 Uncertainty Across OOD Tiers.** Three panels, one per uncertainty component. Each violin shows the distribution of the corresponding component on the molecules of one tier; the black scatter overlays the individual molecules.

The expected pattern is different for each panel:

- **Total $\sigma_*$ (left).** Should shift upward from left to right, ideally monotonically. This is the headline UQ signal the user would consume in deployment &mdash; the single number that says "trust this prediction or not".
- **Aleatoric (middle).** Reflects the ensemble's *predicted* per-input noise level. On OOD molecules the MVE heads were never trained on inputs that look like this, so their aleatoric output extrapolates and can either spike or collapse depending on which direction the variance head was driven during training. A rising trend here is informative but less reliable than the epistemic one.
- **Epistemic (right).** Reflects the spread between ensemble members &mdash; precisely the original deep-ensemble signal. This is typically the *cleanest* OOD detector of the three: on familiar inputs the members agree (low epistemic), on unfamiliar ones they disagree (high epistemic).

A well-calibrated MVE ensemble should show clear stratification on at least the *total* and *epistemic* panels. Disagreement *between* the panels is itself diagnostic: a tier on which epistemic is high but aleatoric is low suggests "the model knows it doesn't know" &mdash; novel chemistry, not noisy chemistry; a tier on which both are high suggests "the model thinks the data is intrinsically noisy here". In our setting we expect Tier 2 (unseen elements) and Tier 3 (PAHs) to be epistemic-dominated, with Tier 1 (heavy molecules) potentially carrying a substantial aleatoric component because heavy molecules in AqSolDB have larger labelling-noise variance to begin with.

<div style="background: #fff0c2; padding: 10px; border-style: solid; border-width: 1px; border-color: #dbad21; border-radius: 3px; color: black;">

**🛠️ Exercise 4.4** $\cdot$ For Tier 3 (PAHs), examine the relationship between PAH size (number of heavy atoms) and ensemble uncertainty. Does uncertainty grow monotonically with size, or does it saturate? The `pahs_df` DataFrame already carries an `n_heavy_atoms` column &mdash; plot it on the x-axis against the per-molecule total/epistemic uncertainty to inspect.

</div>

### Latent-Space Geometry of OOD Tiers

So far we have measured "how far OOD" each tier sits indirectly &mdash; via the magnitude of the ensemble's uncertainty signal. A more direct diagnostic is to look at where each tier ends up in the model's *learned feature space*. If the OOD tiers really do sit outside the training distribution, they should also be visibly separated from the training set's cloud in the GNN's pooled graph-embedding space &mdash; not blended into the middle of it.

We use the first ensemble member to produce a graph-level embedding (`_graph_embedding(...)`) for every training molecule and every molecule across the four tiers. We then fit a `UMAP` reducer on the *training set embeddings only* and use that same fitted mapper to project the OOD tiers. Crucially, the UMAP fit never sees the OOD molecules &mdash; so any separation we see in the 2-D plot is geometry the model itself put there, not something UMAP carved out to make the picture pretty.

Alongside the 2-D plot we also report the per-tier distribution of *k-nearest-neighbour distances* in the original (high-dimensional) embedding space: for every OOD molecule we measure its mean distance to the 5 nearest training-set embeddings. A larger kNN distance is the high-dimensional analogue of "lies outside the training cloud".

In [ ]:
# --- Extract graph embeddings, fit UMAP on the training set, project everything ---

import umap
from sklearn.neighbors import NearestNeighbors


@torch.no_grad()
def graph_embeddings_for(smiles_list: list[str], model: SolubilityGNN) -> np.ndarray:
    """
    Compute the pooled graph-level embedding (the hidden vector that goes into the
    mean and variance heads) for every SMILES in `smiles_list`, using one ensemble
    member. SMILES that fail to parse are skipped.

    :return: Array of shape (n_valid, hidden_dim).
    """
    model.eval()
    embs: list[np.ndarray] = []
    for smi in smiles_list:
        try:
            graph = graph_from_smiles(smi)
        except Exception:
            continue
        data = data_from_graph(graph)
        data.batch = torch.zeros(data.x.shape[0], dtype=torch.long)
        h = model._graph_embedding(data).cpu().numpy().reshape(-1)
        embs.append(h)
    return np.stack(embs) if embs else np.zeros((0, model.hparams.hidden_dim), dtype=np.float32)


# We use the first ensemble member as the "reference" encoder. Different ensemble
# members have different hidden_dim and therefore live in different embedding spaces,
# so concatenating their embeddings would be meaningless.
encoder = ensemble[0]

print(f'Using ensemble member 1 as embedding encoder (hidden_dim={encoder.hparams.hidden_dim})')

# Embed training set + every tier
train_smiles = data_frame['smiles'].iloc[train_indices].tolist()
print('Computing graph embeddings...')
emb_train = graph_embeddings_for(train_smiles, encoder)
emb_t0    = graph_embeddings_for(test_smiles, encoder)         # in-dist test (tier 0)
emb_t1    = graph_embeddings_for(tier1_smiles, encoder)        # heavy
emb_t2    = graph_embeddings_for(tier3_smiles, encoder)        # unseen element
emb_t3    = graph_embeddings_for(tier4_smiles, encoder)        # PAHs
print(f'  train: {emb_train.shape}, t0: {emb_t0.shape}, t1: {emb_t1.shape}, '
      f't2: {emb_t2.shape}, t3: {emb_t3.shape}')

# --- Fit UMAP on the training-set embeddings only ---
print('Fitting UMAP on training-set embeddings...')
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
proj_train = reducer.fit_transform(emb_train)
proj_t0    = reducer.transform(emb_t0)
proj_t1    = reducer.transform(emb_t1)
proj_t2    = reducer.transform(emb_t2)
proj_t3    = reducer.transform(emb_t3)


# --- kNN distance to the training set in the *original* embedding space ---
# Mean distance to the 5 nearest training neighbours. Computed in the un-reduced
# embedding to avoid UMAP-induced distortion.

K_NN = 5
nn_search = NearestNeighbors(n_neighbors=K_NN, algorithm='auto').fit(emb_train)

def mean_knn_distance(emb: np.ndarray) -> np.ndarray:
    if len(emb) == 0:
        return np.array([])
    dist, _ = nn_search.kneighbors(emb, n_neighbors=K_NN, return_distance=True)
    return dist.mean(axis=1)

knn_train = mean_knn_distance(emb_train)   # self-distances; the nearest neighbour will be the point itself (distance 0)
knn_t0 = mean_knn_distance(emb_t0)
knn_t1 = mean_knn_distance(emb_t1)
knn_t2 = mean_knn_distance(emb_t2)
knn_t3 = mean_knn_distance(emb_t3)

print('\nMean k=5 NN distance to training set (median over molecules):')
print(f'  train (self):           {np.median(knn_train):.3f}')
print(f'  Tier 0 (in-dist.):      {np.median(knn_t0):.3f}')
print(f'  Tier 1 (heavy):         {np.median(knn_t1):.3f}')
print(f'  Tier 2 (unseen elem.):  {np.median(knn_t2):.3f}')
print(f'  Tier 3 (PAHs):          {np.median(knn_t3):.3f}')


# --- Two-panel figure: UMAP scatter + kNN-distance boxplots ---

fig, (ax_umap, ax_knn) = plt.subplots(1, 2, figsize=(15, 6),
                                       gridspec_kw={'width_ratios': [1.5, 1]})

# Left panel: UMAP scatter, training cloud first (grey), then OOD tiers on top.
ax_umap.scatter(proj_train[:, 0], proj_train[:, 1], s=4, c='lightgrey',
                alpha=0.4, edgecolor='none', label=f'train (n={len(proj_train)})')

cmap = plt.cm.plasma
tier_projs = [proj_t0, proj_t1, proj_t2, proj_t3]
tier_names = ['Tier 0 (in-dist.)', 'Tier 1 (heavy)',
              'Tier 2 (unseen elem.)', 'Tier 3 (PAHs)']
for i, (proj, name) in enumerate(zip(tier_projs, tier_names)):
    col = cmap(i / (len(tier_projs) - 1))
    ax_umap.scatter(proj[:, 0], proj[:, 1], s=22, color=col, alpha=0.85,
                    edgecolor='black', linewidth=0.4, label=f'{name} (n={len(proj)})')

ax_umap.set_xlabel('UMAP dim 1')
ax_umap.set_ylabel('UMAP dim 2')
ax_umap.set_title('Graph-embedding UMAP\n(reducer fit on training set only)')
ax_umap.legend(loc='best', fontsize=8, framealpha=0.9)
ax_umap.grid(alpha=0.3)

# Right panel: violin of kNN distances per tier (excluding self-distances on train).
knn_data = [knn_t0, knn_t1, knn_t2, knn_t3]
knn_labels = ['Tier 0\n(in-dist.)', 'Tier 1\n(heavy)',
              'Tier 2\n(unseen\nelement)', 'Tier 3\n(PAHs)']

parts = ax_knn.violinplot(knn_data, positions=range(len(knn_data)),
                           showmedians=True, widths=0.85)
for i, body in enumerate(parts['bodies']):
    body.set_facecolor(cmap(i / (len(knn_data) - 1)))
    body.set_alpha(0.75)
    body.set_edgecolor('black')
for i, vals in enumerate(knn_data):
    jitter = np.random.uniform(-0.08, 0.08, size=len(vals))
    ax_knn.scatter(np.full_like(vals, i) + jitter, vals, s=6,
                   color='black', alpha=0.35)
# Reference line at the training set's median self-kNN distance
ax_knn.axhline(np.median(knn_train), color='grey', linestyle='--', linewidth=1,
               label=f'train median (k=5)')
ax_knn.set_xticks(range(len(knn_data)))
ax_knn.set_xticklabels(knn_labels, fontsize=9)
ax_knn.set_ylabel(f'Mean k={K_NN} NN distance to train set\n(in raw embedding space)')
# Clip y-axis at the 98th percentile of the combined tier distances so a handful
# of far-OOD outliers don't squash the in-distribution detail.
_all_knn = np.concatenate([v for v in knn_data if len(v) > 0])
if len(_all_knn) > 0:
    _y_hi = np.percentile(_all_knn, 98)
    ax_knn.set_ylim(0, _y_hi * 1.05)
ax_knn.set_title('Embedding-space distance from training cloud')
ax_knn.legend(loc='upper left', fontsize=8)
ax_knn.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

**📊 Latent-Space Geometry.** The left panel shows the GNN's training molecules in light grey and the four tier sets coloured on top, all projected through the same UMAP mapper that was fit *only* on the grey points. The right panel shows, for each tier, the distribution of mean distance to the 5 nearest training-set embeddings &mdash; computed in the *original* high-dimensional embedding space, before UMAP, so that UMAP's known tendency to compress distances does not affect the diagnostic.

What the picture *should* look like for a UQ-friendly encoder:

- **Tier 0 (in-distribution)** should overlap heavily with the grey training cloud on both panels &mdash; same distribution, same embedding cluster, same kNN distance as the training set itself.
- **Tier 1 (heavy)** should sit on the *edge* of the grey cloud, in the size tail; the kNN violin should be visibly above the training baseline.
- **Tier 2 (unseen element)** should fall *outside* the cloud in at least one direction, with a notably elevated kNN distance &mdash; the atom encoder collapses these atoms to "unknown", which gives them a feature signature the encoder has never seen.
- **Tier 3 (PAHs)** should be the most extreme, ideally forming an entirely separate cluster off in its own region of UMAP space with the largest kNN distances.

If instead the OOD tiers fall *within* the training cloud and have kNN distances similar to the training baseline, the encoder has learned to *fold* these molecules onto familiar representations &mdash; which is precisely the failure mode that spectral normalisation and distance-aware feature extractors are designed to suppress. Comparing this latent-space picture against the per-tier uncertainty violins shown earlier is the most direct way to diagnose whether the ensemble's uncertainty signal is geometrically grounded or just a happy accident of the calibration plot.

## **4.4** $\cdot$ 🔬 Discussion and Limitations

Deep ensembles provide a conceptually simple and widely deployed approach to uncertainty quantification in molecular property prediction, but the estimates they produce should be interpreted with several important caveats in mind.

**Aleatoric Uncertainty Is Not Captured.** The ensemble standard deviation reflects how much different members of the ensemble *disagree* about a given prediction. This is fundamentally a measure of *epistemic* uncertainty &mdash; the model's uncertainty about its own parameters, driven by limited training data &mdash; and it tells us essentially nothing about the *aleatoric* uncertainty inherent to the measurement process. On a molecule whose experimental logS value has a known measurement error of $\pm 0.5$, the ensemble may confidently predict the wrong value because all members were trained on the same noisy label. Capturing aleatoric uncertainty requires explicitly predicting a distribution rather than a point estimate (e.g. mean-variance regression) and is not addressed by deep ensembles alone.

**Ensemble Size Matters &mdash; and Three Is Small.** With $N = 3$ members the standard-deviation estimate is itself a small-sample statistic and is correspondingly noisy. The qualitative behaviour shown in this tutorial &mdash; rising uncertainty across OOD tiers, positive correlation with absolute error &mdash; is real, but the absolute numerical values shift noticeably between random seeds and would be sharper at $N = 5$ to $N = 10$. In production deployments where the cost of inference matters less than the reliability of the uncertainty signal, larger ensembles are the norm. The tutorial defaults are chosen for runtime, not for statistical optimality.

**Diversity Is Earned, Not Given.** The whole machinery of deep ensembles relies on the assumption that ensemble members make *different* mistakes. If all members converge to the same solution &mdash; because the subsampling is too mild, the model is too over-parameterised for the dataset, or the training procedure is too aggressive &mdash; the ensemble standard deviation collapses to near-zero and the uncertainty signal disappears. The 80% subsampling used here is a low-cost diversity injection, but more sophisticated ensembles use bootstrap resampling, architectural variation, or even adversarial training to enforce member disagreement. Whether a given ensemble has actually achieved meaningful diversity is itself something practitioners should check &mdash; for instance by verifying that the correlation between uncertainty and error is non-trivially positive, as we did on the in-distribution test set.

**OOD Detection Is a Symptom, Not a Diagnosis.** A rising ensemble uncertainty on an unfamiliar molecule signals that *something* about the input is unusual relative to the training data, but it does not say *what*. The same elevated uncertainty could equally indicate an unseen element, an unusual functional group, or merely an unusually large molecule &mdash; and the appropriate downstream action (discard the prediction, gather more training data, hand off to a human expert) may depend on which of these is the case. Uncertainty quantification should therefore be viewed as a triage signal that prompts further inspection, not as a self-contained diagnosis of model failure.

**Aleatoric Estimates Are Only as Good as Their Coverage.** With MVE the model now does claim an aleatoric uncertainty estimate per input &mdash; but that claim is only meaningful where the variance head saw enough training signal. On in-distribution test molecules the aleatoric estimate is grounded in regions of input space the model has actually observed, and its correlation with |error| can be informative. On out-of-distribution molecules the aleatoric output is itself an extrapolation: the variance head may extrapolate to anomalously *low* values (because nothing in its training experience drove it upward), giving a false impression of confidence. In practice the *epistemic* component &mdash; the spread of per-member means &mdash; remains the more reliable OOD detector, and the *total* $\sigma_*$ should be treated as a sensible-default summary rather than a definitive answer.

## **References**

**Datasets**

- Sorkun, M. C., Khetan, A., & Er, S. (2019). AqSolDB, a curated reference set of aqueous solubility and 2D descriptors for a diverse set of compounds. *Scientific Data*, 6(1), 143. https://doi.org/10.1038/s41597-019-0151-1

**Methods**

- Lakshminarayanan, B., Pritzel, A., & Blundell, C. (2017). Simple and scalable predictive uncertainty estimation using deep ensembles. In *Advances in Neural Information Processing Systems* (NeurIPS), 30. https://arxiv.org/abs/1612.01474

- Gal, Y., & Ghahramani, Z. (2016). Dropout as a Bayesian approximation: Representing model uncertainty in deep learning. In *International Conference on Machine Learning* (ICML), 1050–1059. https://arxiv.org/abs/1506.02142

- Liu, J. Z., Lin, Z., Padhy, S., Tran, D., Bedrax-Weiss, T., & Lakshminarayanan, B. (2020). Simple and principled uncertainty estimation with deterministic deep learning via distance awareness. In *Advances in Neural Information Processing Systems* (NeurIPS), 33, 7498–7512. https://arxiv.org/abs/2006.10108

- Miyato, T., Kataoka, T., Koyama, M., & Yoshida, Y. (2018). Spectral normalization for generative adversarial networks. In *International Conference on Learning Representations* (ICLR). https://arxiv.org/abs/1802.05957

- Xu, K., Hu, W., Leskovec, J., & Jegelka, S. (2019). How powerful are graph neural networks?. In *International Conference on Learning Representations* (ICLR). https://arxiv.org/abs/1810.00826

- Seitzer, M., Tavakoli, A., Antic, D., & Martius, G. (2022). On the pitfalls of heteroscedastic uncertainty estimation with probabilistic neural networks. In *International Conference on Learning Representations* (ICLR). https://arxiv.org/abs/2203.09168

- Nix, D. A., & Weigend, A. S. (1994). Estimating the mean and variance of the target probability distribution. In *Proceedings of the IEEE International Conference on Neural Networks*, 1, 55–60. https://doi.org/10.1109/ICNN.1994.374138

- Teufel, J., Leinweber, A., & Friederich, P. (2025). Improving counterfactual truthfulness for molecular property prediction through uncertainty quantification. In *Explainable Artificial Intelligence (xAI 2025)*. https://github.com/the16thpythonist/truthful_counterfactuals

- Hirschfeld, L., Swanson, K., Yang, K., Barzilay, R., & Coley, C. W. (2020). Uncertainty quantification using neural networks for molecular property prediction. *Journal of Chemical Information and Modeling*, 60(8), 3770–3780. https://doi.org/10.1021/acs.jcim.0c00502

**Software**

- Landrum, G. et al. (2024). RDKit: Open-source cheminformatics software. https://www.rdkit.org

- Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., ... & Chintala, S. (2019). PyTorch: An imperative style, high-performance deep learning library. *Advances in Neural Information Processing Systems*, 32. https://pytorch.org

- Falcon, W. et al. (2019). PyTorch Lightning. https://lightning.ai/docs/pytorch/stable/

- Fey, M., & Lenssen, J. E. (2019). Fast graph representation learning with PyTorch Geometric. In *ICLR Workshop on Representation Learning on Graphs and Manifolds*. https://pytorch-geometric.readthedocs.io